# TXXX · Rigid and Flexible Docking with GNINA

Authors:

 Andrew Zaki, 2024, Saarland University
 
 Denis Semkin, 2024, Saarland University

The examples used in this talktorial template are taken from [T015](https://projects.volkamerlab.org/teachopencadd/talktorials/T015_protein_ligand_docking.html)


# Aim of this talktorial

The goal of this study is to investigate and develop molecular docking procedures, with a particular emphasis on the implications of protein and ligand flexibility on docking accuracy. We use GNINA to illustrate both rigid-protein/flexible-ligand docking and flexible-protein/flexible-ligand docking approaches. The work attempts to:

1- Provide a step-by-step approach to creating protein and ligand structures for docking.

2- Examine how protein flexibility influences docking outcomes in both redocking and cross-docking conditions.

3- Use RMSD calculations to determine docking accuracy in relation to experimental crystal structures.

4- Describe the computational trade-offs of rigid and flexible docking.

5- Provide insight into the practical applications of docking techniques in drug development and structure-based research.

# Contents in Theory
## Table of Contents
- [Introduction](#Introduction)
- [Key Concepts](#Key-Concepts)
- [Simulation Workflow](#Simulation-Workflow)


# Introduction

 - Molecular docking - process of predicting the orientation, position, and binding affinity of a ligand as it interacts with a target receptor.
- This process mimics the natural interactions that occur between biomolecules, such as a drug interacting with an enzyme or receptor in the body.
- There are two types of docking process : Rigid(classical)docking and flexible docking
- Rigid docking Rigid keeps the protein static while allowing ligand flexibility. Flexible docking allows certain residues or regions of the protein to adapt, enabling better exploration of binding interactions but increasing computational cost.
- RMSD as a Metric : RMSD measures the deviation between docked poses and reference crystal structures. RMSD < 2 Å is typically considered a successful reproduction of binding poses.
# Key Concepts
- **Molecular Flexibility**:
  - Proteins are not rigid; they undergo conformational changes to optimize interactions with ligands (induced fit hypothesis).
  - Ligands, especially small molecules, can explore numerous conformations due to bond rotations, allowing better complementarity to the protein binding site.
    
- **Degrees of Freedom**:
  - Flexible docking involves sampling multiple torsional angles for the ligand and accounting for side-chain and/or backbone movements in the protein.
  - While this adds complexity, it significantly improves docking accuracy.

# Simulation Workflow
1. **Ligand Preparation**:
   - Ligand flexibility is introduced by allowing bond rotations and conformational sampling.
   - Rotatable bonds and dihedral angles are defined for dynamic adjustments during docking.
2. **Protein Preparation**:
   - Define the side-chain flexibility: Commonly used for residues near the binding site.
4. **Search Algorithm**:
   - Algorithms such as Monte Carlo, Genetic Algorithms, or Molecular Dynamics are used to explore the conformational space of both the ligand and the protein.
   - Scoring functions evaluate binding affinity for each sampled conformation.
5. **Result Evaluation**:
   - Docking results typically consist of clusters of conformations ranked by predicted binding affinity and stability.
   - Analysis often includes identifying the most plausible binding mode and key molecular interactions (e.g., hydrogen bonds, hydrophobic contacts).


# Practice

1. **Implement flexible-protein / flexible-ligand docking with GNINA**

2. **Fetching of PDB files**

3. **Alignment of fetched proteins**

4. **Extraction of ligands from aligned PDB files**

5. **Selection of binding sites** 

6. **Preparation of proteins using PDBFixer**

7. **Preparation of ligands** 

8. **Docking into selected binding sites** 

9. **Calculation of RMSD relative to crystal ligands** 

10. **RMSD between rigid and flexible ligand poses** 

11. **Comparison of position/coordinates of protein residues (develop on your own)** 

In [1]:
### INSTALL IF NECCESSARY ###
# import locale
# locale.getpreferredencoding = lambda: "UTF-8"
# !nvidia-smi
# !wget https://github.com/gnina/gnina/releases/download/v1.3/gnina
# !chmod +x gnina
# !./gnina --version
# # !pip install -q condacolab ### if running in colab ###
# # import condacolab
# # condacolab.install()
# # !conda --version
# !conda install conda-forge::pdbfixer
# !conda install conda-forge::nglview
# !conda install conda-forge::openbabel
# !conda install conda-forge::mdanalysis
# !conda install conda-forge::rdkit
# !conda install conda-forge::py3dmol



In [2]:
import os
import requests
import subprocess
import pandas as pd
import py3Dmol
import subprocess

from pdbfixer import PDBFixer
from openmm.app import PDBFile
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolAlign
from Bio.PDB import PDBParser, Superimposer, PDBIO
from Bio.PDB.cealign import CEAligner
from openbabel import pybel


PDB_DIR = "pdb_files/"
LIGAND_DIR = "ligands/"
DOCKING_DIR = "docking_results/"
os.makedirs(PDB_DIR, exist_ok=True)
os.makedirs(LIGAND_DIR, exist_ok=True)
os.makedirs(DOCKING_DIR, exist_ok=True)

!mkdir rmsd_values

## Fetch PDB files
Fetch the PDB files for 4IAQ and 6G79 and repair missing atoms or residues using PDBFixer

In [3]:
# --- 1. Fetch PDB Files ---

def fetch_pdb(pdb_id):
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    response = requests.get(url)
    pdb_path = f"{PDB_DIR}{pdb_id}.pdb"
    
    if response.status_code == 200:
        with open(pdb_path, 'wb') as f:
            f.write(response.content)
        print(f"{pdb_id} downloaded.")
        return pdb_path
    else:
        raise Exception(f"Failed to fetch PDB {pdb_id}")
    
# Fetch and prepare proteins
pdb_4iaq = fetch_pdb("4IAQ")
pdb_6g79 = fetch_pdb("6G79")

4IAQ downloaded.
6G79 downloaded.


## Align Proteins
Align protein structures to ensure consistency in docking experiments

In [4]:
# --- 2. Align Protein Structures ---
def align_pdbs(reference_pdb, target_pdb):

    # Initialize PDB parser
    parser = PDBParser(QUIET=True)

    # Parse the structures
    ref_structure = parser.get_structure("ref", reference_pdb)
    target_structure = parser.get_structure("target", target_pdb)
    
    # Initialize CEAligner
    aligner = CEAligner()
    aligner.set_reference(ref_structure)
    
    # Perform CE alignment
    aligner.align(target_structure)

    # Save the aligned structures
    io = PDBIO()
    io.set_structure(ref_structure)
    io.save('pdb_files/4IAQ_aligned.pdb')

    io.set_structure(target_structure)
    io.save('pdb_files/6G79_aligned.pdb')
    
# Align proteins
align_pdbs(f"{PDB_DIR}4IAQ.pdb", f"{PDB_DIR}6G79.pdb")

In [5]:
# ## Check the alignment ###

# # Create 3Dmol.js viewer
# v = py3Dmol.view(width=1000, height=800)

# #Load protein structure
# v.addModel(open('pdb_files/4IAQ_aligned.pdb').read())  # Protein 1
# v.setStyle({'cartoon': {}, 'stick': {'radius': .1}})

# v.addModel(open('pdb_files/6G79_aligned.pdb').read())  # Protein 2
# v.setStyle({'cartoon': {}, 'stick': {'radius': .1}})

# v.rotate(90)  # Rotate for better visualization
# v.show()

### Extract true ligands (for RMSD calc later)

In [6]:
!grep 2GM pdb_files/4IAQ_aligned.pdb > ligands/true_2GM_lig.pdb

!grep EP5 pdb_files/6G79_aligned.pdb > ligands/true_EP5_lig.pdb

## Select Binding Sites
Use the co-crystallized ligand to define the binding site.

Calculate the center and size of the binding pocket

In [7]:
# --- 3. Select Binding Site ---

def select_binding_site(pdb_file, ligand_resname):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure("protein", pdb_file)
    
    ligand_atoms = []
    for model in structure:
        for chain in model:
            for res in chain:
                if res.resname == ligand_resname:
                    for atom in res:
                        ligand_atoms.append(atom.coord)
    
    if not ligand_atoms:
        raise ValueError(f"Ligand {ligand_resname} not found in {pdb_file}")

    # Calculate center of mass for the ligand binding site
    center = sum(ligand_atoms) / len(ligand_atoms)
    print(f"Binding site center for {pdb_file}: {center}")
    return center

# Binding site selection
center_4iaq = select_binding_site(f"{PDB_DIR}4IAQ_aligned.pdb", "2GM")
center_6g79 = select_binding_site(f"{PDB_DIR}6G79_aligned.pdb", "EP5")

Binding site center for pdb_files/4IAQ_aligned.pdb: [-21.06556   10.188115  21.857306]
Binding site center for pdb_files/6G79_aligned.pdb: [-20.772429   9.942136  24.59037 ]


### After manual inspection of binding pockets in PyMol, site coordinates and sizes were altered to better suit the experiment


In [8]:
center_4iaq = [-21.06, 8.48, 21.85]
size_4iaq = [30, 31, 31]
center_6g79 = [-18.56, 8.48, 25.05]
size_6g79 = [32, 31, 32]

## Prepare the protein using PDBFixer

In [9]:
# --- 4. Prepare Protein with PDBFixer ---

def prepare_protein(input_pdb, output_pdb):
    fixer = PDBFixer(filename=input_pdb)
    fixer.findMissingResidues()
    fixer.findNonstandardResidues()
    fixer.replaceNonstandardResidues()
    fixer.removeHeterogens(True)
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(7.0)
    with open(output_pdb, 'w') as f:
        PDBFile.writeFile(fixer.topology, fixer.positions, f)
    
    print(f"Protein {input_pdb} prepared as {output_pdb}")
    
prepare_protein(f"{PDB_DIR}4IAQ_aligned.pdb", f"{PDB_DIR}4IAQ_prepared.pdb")
prepare_protein(f"{PDB_DIR}6G79_aligned.pdb", f"{PDB_DIR}6G79_prepared.pdb")


Protein pdb_files/4IAQ_aligned.pdb prepared as pdb_files/4IAQ_prepared.pdb
Protein pdb_files/6G79_aligned.pdb prepared as pdb_files/6G79_prepared.pdb


## Prepare Ligands from SMILES

In [10]:
# --- 5. Prepare Ligand from SMILES (.sdf format) ---

def prepare_ligand_from_smiles(smiles, ligand_name):
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol)
    ligand_path = f"{LIGAND_DIR}{ligand_name}.sdf"
    
    writer = Chem.SDWriter(ligand_path)
    writer.write(mol)
    writer.close()
    
    print(f"Ligand {ligand_name} saved as {ligand_path}")
    return ligand_path

ligand_2gm = prepare_ligand_from_smiles("CN1C[C@@H](C[C@H]2[C@H]1Cc3c[nH]c4cccc2c34)C(=O)N[C@]5(C)O[C@@]6(O)[C@@H]7CCCN7C(=O)[C@H](Cc8ccccc8)N6C5=O", "2GM")  # Example SMILES
ligand_ep5 = prepare_ligand_from_smiles("[NH3+]CCc1c[nH]c2ccc(OCC(=O)N3CCN(CC3)c4ccc(cc4)C#N)cc12", "EP5")

Ligand 2GM saved as ligands/2GM.sdf
Ligand EP5 saved as ligands/EP5.sdf


# Redocking
### We will utilize GNINA to perform redocking of ligands 2GM and EP5 with their proteins 4IAQ and 6G79 respectively.

## 4IAQ(rigid protein)+2GM(flexible ligand)

In [11]:
# --- Perform Docking with GNINA ---

def perform_docking(protein_pdb, ligand_sdf, center, output_sdf, size, exhaustiveness=8, seed=123):
    """
    Perform docking using GNINA with a specified center and grid size.

    Args:
        protein_pdb (str): Path to the prepared protein PDB file.
        ligand_sdf (str): Path to the ligand SDF file.
        center (tuple): Coordinates for the docking box center (x, y, z).
        output_sdf (str): Path to the output SDF file.
        size (tuple): Dimensions of the docking grid (size_x, size_y, size_z).
        exhaustiveness (int): GNINA exhaustiveness parameter.
        seed (int): GNINA seed for reproducibility.
    """
    # Ensure output directories exist
    output_dir = os.path.dirname(output_sdf)
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Build the GNINA command
    command = [
        "gnina",
        "-r", protein_pdb,
        "-l", ligand_sdf,
        "--center_x", str(center[0]),
        "--center_y", str(center[1]),
        "--center_z", str(center[2]),
        "--size_x", str(size[0]),
        "--size_y", str(size[1]),
        "--size_z", str(size[2]),
        "-o", output_sdf,
        "--exhaustiveness", str(exhaustiveness),
        "--seed", str(seed)
    ]
    
    # Run the command and capture the result
    result = subprocess.run(command, capture_output=True, text=True)
    
    # Check if the command was successful
    if result.returncode == 0:
        print(f"Docking complete: {output_sdf}")
    else:
        print(f"Error during docking: {result.stderr}")
    
    return output_sdf

# Example usage
protein_pdb = f"{PDB_DIR}4IAQ_prepared.pdb"
ligand_sdf = ligand_2gm
center = center_4iaq
size = size_4iaq
output_sdf = "./docking_results/4IAQ_2GM_docked.sdf"

docked_4iaq_2gm = perform_docking(
    protein_pdb=protein_pdb,
    ligand_sdf=ligand_sdf,
    center=center,
    size=size,
    output_sdf=output_sdf)


Docking complete: ./docking_results/4IAQ_2GM_docked.sdf


In [12]:
# --- Calculate RMSD ---
!obrms -f ligands/true_2GM_lig.pdb docking_results/4IAQ_2GM_docked.sdf > rmsd_values/4iaq_2gm_rmsd.txt



mkdir: cannot create directory ‘rmsd_values’: File exists
*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is ligands/true_2GM_lig.pdb)



In [13]:
def parse_rmsd_file(rmsd_file):
    """
    Parse the RMSD output file to find the index of the lowest RMSD value.
    
    Args:
        rmsd_file (str): Path to the RMSD output file.
    
    Returns:
        int: Index of the pose with the lowest RMSD.
        float: The lowest RMSD value.
    """
    rmsd_values = []
    with open(rmsd_file, 'r') as f:
        for line in f:
            # Extract RMSD values from each line
            if "RMSD" in line:
                rmsd = float(line.split(":")[-1].strip())
                rmsd_values.append(rmsd)
    
    # Find the index of the minimum RMSD value
    min_rmsd_index = rmsd_values.index(min(rmsd_values))
    return min_rmsd_index, rmsd_values[min_rmsd_index]

parse_rmsd_file("rmsd_values/4iaq_2gm_rmsd.txt")


(1, 0.964025)

In [14]:
import py3Dmol
from rdkit import Chem

def show_specific_pose(orig_prot, orig_lig, docking_lig, pose_index):
    """
    Visualize a specific pose from an SDF file using Py3Dmol.
    
    Args:
        sdf_file (str): Path to the input SDF file.
        pose_index (int): Index of the pose to visualize.
    
    Returns:
        py3Dmol.view: A Py3Dmol view object for visualization.
    """
    # Load molecules from the SDF file
    suppl = Chem.SDMolSupplier(docking_lig)
    if not suppl or pose_index >= len(suppl):
        raise ValueError("Invalid SDF file or index out of range.")
    
    # Extract the specific pose
    selected_pose = suppl[pose_index]
    if selected_pose is None:
        raise ValueError(f"Pose at index {pose_index} is invalid or corrupted.")
    
    # Convert the pose to a MOL block for visualization
    mol_block = Chem.MolToMolBlock(selected_pose)
    
    # Initialize Py3Dmol view
    view = py3Dmol.view()
    view.addModel(mol_block, 'mol')  # Add the pose as a model
    view.setStyle({'stick': {}})     # Set the visualization style
    view.zoomTo()                    # Zoom to fit the model
    return view

# Example usage
docking_lig = "docking_results/4IAQ_2GM_docked.sdf"
pose_index = 1  # Index of the pose to visualize

view = show_specific_pose(docking_lig, pose_index)
view.show()


TypeError: show_specific_pose() missing 2 required positional arguments: 'docking_lig' and 'pose_index'

In [15]:
### all modes
def show_all_modes(orig_prot, orig_lig, docking_lig):
    v = py3Dmol.view()
    v.addModel(open(orig_prot).read())
    v.setStyle({'cartoon':{},'stick':{'radius':.1}})
    v.addModel(open(orig_lig).read())
    v.setStyle({'model':1},{'stick':{'colorscheme':'dimgrayCarbon','radius':.125}})
    v.addModelsAsFrames(open(docking_lig).read())
    v.setStyle({'model':2},{'stick':{'colorscheme':'greenCarbon'}})
    v.zoomTo({'model':1})
    v.rotate(90)
    v.animate({'interval':2000})
    return v

show_all_modes(orig_prot='pdb_files/4IAQ_prepared.pdb', orig_lig='ligands/true_2GM_lig.pdb', docking_lig='docking_results/4IAQ_2GM_docked.sdf')

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 4IAQ(flexible protein)+2GM(flexible ligand)

In [17]:
# --- Perform Docking with GNINA ---

def perform_flex_docking(protein_pdb, ligand_sdf, center, output_sdf, flexdist, flexdist_ligand, out_flex, size, exhaustiveness=8, seed=123):

    # Ensure output directories exist
    output_dir = os.path.dirname(output_sdf)
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
    
    # Build the GNINA command
    command = [
        "gnina",
        "-r", protein_pdb,
        "-l", ligand_sdf,
        "--center_x", str(center[0]),
        "--center_y", str(center[1]),
        "--center_z", str(center[2]),
        "--size_x", str(size[0]),
        "--size_y", str(size[1]),
        "--size_z", str(size[2]),
        "-o", output_sdf,
        "--flexdist", str(flexdist),
        "--flexdist_ligand", flexdist_ligand,
        "--out_flex", out_flex,
        "--exhaustiveness", str(exhaustiveness),
        "--seed", str(seed)
    ]
    
    # Run the command and capture the result
    result = subprocess.run(command, capture_output=True, text=True)
    
    # Check if the command was successful
    if result.returncode == 0:
        print(f"Docking complete: {output_sdf}")
    else:
        print(f"Error during docking: {result.stderr}")
    
    return output_sdf

# Example usage
protein_pdb = f"{PDB_DIR}4IAQ_prepared.pdb"
ligand_sdf = ligand_2gm
center = center_4iaq
size = size_4iaq
output_sdf = "./docking_results/flex_4IAQ_2GM_docked.sdf"
flexdist = 8
flexdist_ligand = "ligands/true_2GM_lig.pdb"
out_flex = "flexout2_4IAQ.pdb"

docked_flex_4iaq_2gm = perform_flex_docking(
    protein_pdb=protein_pdb,
    ligand_sdf=ligand_sdf,
    center=center,
    size=size,
    output_sdf=output_sdf,
    flexdist=flexdist,
    flexdist_ligand=flexdist_ligand,
    out_flex=out_flex
)


Docking complete: ./docking_results/flex_4IAQ_2GM_docked.sdf


In [18]:
# --- Calculate RMSD ---

!obrms -f ligands/true_2GM_lig.pdb docking_results/flex_4IAQ_2GM_docked.sdf

*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is ligands/true_2GM_lig.pdb)

RMSD ligands/true_2GM_lig.pdb: 2.6184
RMSD ligands/true_2GM_lig.pdb: 0.964025
RMSD ligands/true_2GM_lig.pdb: 4.22042
RMSD ligands/true_2GM_lig.pdb: 9.03429
RMSD ligands/true_2GM_lig.pdb: 3.99393
RMSD ligands/true_2GM_lig.pdb: 7.1556
RMSD ligands/true_2GM_lig.pdb: 10.7584
RMSD ligands/true_2GM_lig.pdb: 7.15811
RMSD ligands/true_2GM_lig.pdb: 5.92606


In [19]:
#show_all_modes(orig_prot='pdb_files/4IAQ_prepared.pdb', orig_lig='ligands/true_2GM_lig.pdb', docking_lig='docking_results/flex_4IAQ_2GM_docked.sdf')

## 6G79(rigid protein)+EP5(flexible ligand)

In [20]:
# --- Perform Docking with GNINA ---

protein_pdb = f"{PDB_DIR}6G79_prepared.pdb"
ligand_sdf = ligand_ep5
center = center_6g79
size = size_6g79
output_sdf = "./docking_results/6G79_EP5_docked.sdf"

docked_6g79_ep5 = perform_docking(
    protein_pdb=protein_pdb,
    ligand_sdf=ligand_sdf,
    center=center,
    size=size,
    output_sdf=output_sdf
)


Docking complete: ./docking_results/6G79_EP5_docked.sdf


In [21]:
# --- Calculate RMSD ---

!obrms -f ligands/true_EP5_lig.pdb docking_results/6G79_EP5_docked.sdf

*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is ligands/true_EP5_lig.pdb)

RMSD ligands/true_EP5_lig.pdb: 1.26439
RMSD ligands/true_EP5_lig.pdb: 1.52103
RMSD ligands/true_EP5_lig.pdb: 2.52924
RMSD ligands/true_EP5_lig.pdb: 2.04631
RMSD ligands/true_EP5_lig.pdb: 10.3235
RMSD ligands/true_EP5_lig.pdb: 2.43652
RMSD ligands/true_EP5_lig.pdb: 2.00925
RMSD ligands/true_EP5_lig.pdb: 2.65179
RMSD ligands/true_EP5_lig.pdb: 5.91294


In [22]:
#show_all_modes(orig_prot="pdb_files/6G79_prepared.pdb", orig_lig='ligands/true_EP5_lig.pdb', docking_lig="docking_results/6G79_EP5_docked.sdf")

## 6G79(flexible protein)+EP5(flexible ligand)

In [23]:
# Example usage
protein_pdb = f"{PDB_DIR}6G79_prepared.pdb"
ligand_sdf = ligand_ep5
center = center_6g79
size = size_6g79
output_sdf = "./docking_results/flex_6G79_EP5_docked.sdf"
flexdist = 8
flexdist_ligand = "ligands/true_EP5_lig.pdb"
out_flex = "flexout4_6G79.pdb"



docked_flex_6g79_ep5= perform_flex_docking(
    protein_pdb=protein_pdb,
    ligand_sdf=ligand_sdf,
    center=center,
    size=size,
    output_sdf=output_sdf,
    flexdist=flexdist,
    flexdist_ligand=flexdist_ligand,
    out_flex=out_flex
)


Docking complete: ./docking_results/flex_6G79_EP5_docked.sdf


In [24]:
# --- Calculate RMSD ---

!obrms -f ligands/true_EP5_lig.pdb docking_results/flex_6G79_EP5_docked.sdf

*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is ligands/true_EP5_lig.pdb)

RMSD ligands/true_EP5_lig.pdb: 1.26439
RMSD ligands/true_EP5_lig.pdb: 1.52103
RMSD ligands/true_EP5_lig.pdb: 2.52924
RMSD ligands/true_EP5_lig.pdb: 2.04631
RMSD ligands/true_EP5_lig.pdb: 10.3235
RMSD ligands/true_EP5_lig.pdb: 2.43652
RMSD ligands/true_EP5_lig.pdb: 2.00925
RMSD ligands/true_EP5_lig.pdb: 2.65179
RMSD ligands/true_EP5_lig.pdb: 5.91294


In [25]:
#show_all_modes(orig_prot="pdb_files/6G79_prepared.pdb", orig_lig='ligands/true_EP5_lig.pdb', docking_lig="docking_results/flex_6G79_EP5_docked.sdf")

### The results of re-docking experiments with rigid and flexible proteins are identical, as the binding pocket is already in its optimal conformation to accommodate the ligand effectively.

# Cross-Docking
### We will utilize GNINA to perform crossdocking of ligands EP5 and 2GM with proteins 4IAQ and 6G79 respectively.

## 4IAQ(rigid protein)+EP5(flexible ligand)

In [26]:
# --- Perform Docking with GNINA ---

protein_pdb = f"{PDB_DIR}4IAQ_prepared.pdb"
ligand_sdf = ligand_ep5
center = center_4iaq
size = size_4iaq
output_sdf = "./docking_results/4IAQ_EP5_docked.sdf"

docked_4iaq_ep5 = perform_docking(
    protein_pdb=protein_pdb,
    ligand_sdf=ligand_sdf,
    center=center,
    size=size,
    output_sdf=output_sdf
)


Docking complete: ./docking_results/4IAQ_EP5_docked.sdf


In [27]:
# --- Calculate RMSD ---

!obrms -f ligands/true_EP5_lig.pdb docking_results/4IAQ_EP5_docked.sdf

*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is ligands/true_EP5_lig.pdb)

RMSD ligands/true_EP5_lig.pdb: 3.50756
RMSD ligands/true_EP5_lig.pdb: 4.39909
RMSD ligands/true_EP5_lig.pdb: 1.97553
RMSD ligands/true_EP5_lig.pdb: 2.66899
RMSD ligands/true_EP5_lig.pdb: 2.75136
RMSD ligands/true_EP5_lig.pdb: 4.30503
RMSD ligands/true_EP5_lig.pdb: 2.84907
RMSD ligands/true_EP5_lig.pdb: 7.27333
RMSD ligands/true_EP5_lig.pdb: 5.29152


In [40]:
show_all_modes(orig_prot="pdb_files/4IAQ_prepared.pdb", orig_lig='ligands/true_EP5_lig.pdb', docking_lig="docking_results/4IAQ_EP5_docked.sdf")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 4IAQ(flexible protein)+EP5(flexible ligand)

In [29]:
# --- Perform Docking with GNINA ---

protein_pdb = f"{PDB_DIR}4IAQ_prepared.pdb"
ligand_sdf = ligand_ep5
center = center_4iaq
size = size_4iaq
output_sdf = "./docking_results/flex_4IAQ_EP5_docked.sdf"
flexdist = 8
flexdist_ligand = "ligands/true_2GM_lig.pdb"
out_flex = "flexout6_4IAQ.pdb"



docked_flex_4iaq_ep5= perform_flex_docking(
    protein_pdb=protein_pdb,
    ligand_sdf=ligand_sdf,
    center=center,
    size=size,
    output_sdf=output_sdf,
    flexdist=flexdist,
    flexdist_ligand=flexdist_ligand,
    out_flex=out_flex
)

Docking complete: ./docking_results/flex_4IAQ_EP5_docked.sdf


In [30]:
# --- Calculate RMSD ---

!obrms -f ligands/true_EP5_lig.pdb docking_results/flex_4IAQ_EP5_docked.sdf

*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is ligands/true_EP5_lig.pdb)

RMSD ligands/true_EP5_lig.pdb: 3.50756
RMSD ligands/true_EP5_lig.pdb: 4.39909
RMSD ligands/true_EP5_lig.pdb: 1.97553
RMSD ligands/true_EP5_lig.pdb: 2.66899
RMSD ligands/true_EP5_lig.pdb: 2.75136
RMSD ligands/true_EP5_lig.pdb: 4.30503
RMSD ligands/true_EP5_lig.pdb: 2.84907
RMSD ligands/true_EP5_lig.pdb: 7.27333
RMSD ligands/true_EP5_lig.pdb: 5.29152


In [39]:
show_all_modes(orig_prot="pdb_files/4IAQ_prepared.pdb", orig_lig='ligands/true_EP5_lig.pdb', docking_lig="docking_results/flex_4IAQ_EP5_docked.sdf")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 6G79(rigid protein)+2GM(flexible ligand)

In [32]:
# --- Perform Docking with GNINA ---

protein_pdb = f"{PDB_DIR}6G79_prepared.pdb"
ligand_sdf = ligand_2gm
center = center_6g79
size = size_6g79
output_sdf = "./docking_results/6G79_2GM_docked.sdf"

docked_6g79_2gn = perform_docking(
    protein_pdb=protein_pdb,
    ligand_sdf=ligand_sdf,
    center=center,
    size=size,
    output_sdf=output_sdf
)


Docking complete: ./docking_results/6G79_2GM_docked.sdf


In [33]:
# --- Calculate RMSD ---

!obrms -f ligands/true_2GM_lig.pdb docking_results/6G79_2GM_docked.sdf

*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is ligands/true_2GM_lig.pdb)

RMSD ligands/true_2GM_lig.pdb: 18.0118
RMSD ligands/true_2GM_lig.pdb: 18.0451
RMSD ligands/true_2GM_lig.pdb: 17.5677
RMSD ligands/true_2GM_lig.pdb: 19.9545
RMSD ligands/true_2GM_lig.pdb: 19.4145
RMSD ligands/true_2GM_lig.pdb: 19.3864
RMSD ligands/true_2GM_lig.pdb: 17.7624
RMSD ligands/true_2GM_lig.pdb: 17.1322
RMSD ligands/true_2GM_lig.pdb: 18.1242


In [34]:
#show_all_modes(orig_prot="pdb_files/6G79_prepared.pdb", orig_lig='ligands/true_2GM_lig.pdb', docking_lig="docking_results/6G79_2GM_docked.sdf")

## 6G79(flexible protein)+2GM(flexible ligand)

In [35]:
# --- Perform Docking with GNINA ---

protein_pdb = f"{PDB_DIR}6G79_prepared.pdb"
ligand_sdf = ligand_2gm
center = center_6g79
size = size_6g79
output_sdf = "./docking_results/flex_6G79_2GM_docked.sdf"
flexdist = 8
flexdist_ligand = "ligands/true_EP5_lig.pdb"
out_flex = "flexout8_6G79.pdb"



docked_flex_6g79_2gm= perform_flex_docking(
    protein_pdb=protein_pdb,
    ligand_sdf=ligand_sdf,
    center=center,
    size=size,
    output_sdf=output_sdf,
    flexdist=flexdist,
    flexdist_ligand=flexdist_ligand,
    out_flex=out_flex
)

Docking complete: ./docking_results/flex_6G79_2GM_docked.sdf


In [36]:
# --- Calculate RMSD ---

!obrms -f ligands/true_2GM_lig.pdb docking_results/flex_6G79_2GM_docked.sdf

*** Open Babel Warning  in PerceiveBondOrders
  Failed to kekulize aromatic bonds in OBMol::PerceiveBondOrders (title is ligands/true_2GM_lig.pdb)

RMSD ligands/true_2GM_lig.pdb: 18.0118
RMSD ligands/true_2GM_lig.pdb: 18.0451
RMSD ligands/true_2GM_lig.pdb: 17.5677
RMSD ligands/true_2GM_lig.pdb: 19.9545
RMSD ligands/true_2GM_lig.pdb: 19.4145
RMSD ligands/true_2GM_lig.pdb: 19.3864
RMSD ligands/true_2GM_lig.pdb: 17.7624
RMSD ligands/true_2GM_lig.pdb: 17.1322
RMSD ligands/true_2GM_lig.pdb: 18.1242


In [37]:
show_all_modes(orig_prot="pdb_files/6G79_prepared.pdb", orig_lig='ligands/true_2GM_lig.pdb', docking_lig="docking_results/flex_6G79_2GM_docked.sdf")

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [38]:
v = py3Dmol.view(height=340,width=940)
v.addModel(open('pdb_files/6G79_prepared.pdb').read())
v.setStyle({'model':0},{'cartoon':{'colorscheme':'dimgrayCarbon'},'stick':{'radius':.1,'colorscheme':'dimgrayCarbon'}})
v.addModel(open('docking_results/flex_6G79_2GM_docked.sdf').read())
v.setStyle({'model':1},{'stick':{'colorscheme':'magentaCarbon'}})
v.addModel(open('flexout8_6G79.pdb').read())
v.setStyle({'model':2},{'stick':{'colorscheme':'magentaCarbon'}})
v.zoomTo({'model':1}); v.rotate(90,'x')

3Dmol.js failed to load for some reason. Please check your browser console for error messages.